In [ ]:
%matplotlib widget

In [ ]:
import hyperspy.api as hs
import os
import zarr

# Get a list of all dm3 files

In [ ]:
main_folder = "/cluster/projects/itea_lille-nv-fys/TFY4330_nanotools_2026/Gruppe3/"
all_files = os.listdir(main_folder)
dm3_files_sorted = []

for file in all_files:
    if file[-4:] == ".dm3":
        dm3_files_sorted.append(file)
dm3_files_sorted.remove("test.dm3")

def sort_dm3(dm3):
    return int(dm3[:3])
dm3_files_sorted.sort(key=sort_dm3)
print(dm3_files_sorted)
print()

dm3_files_sorted_1 = dm3_files_sorted[:dm3_files_sorted.index("023.dm3")+1]
dm3_files_sorted_2 = dm3_files_sorted[dm3_files_sorted.index("023.dm3")+1:]
print(dm3_files_sorted_1)

# Get a list of all mib files

Here I used AI. AI is really good at specifically making scripts like this and I am lazy.

Tip for AI: the `tree` command is useful to get the filestructure. Just pipe the result into a text file and upload it as context to the LLM. You can open a linux shell directly in jupyter lab.

In [ ]:
# Map each .mib filename to its path relative to main_folder
mib_by_name = {}

for root, directories, files in os.walk(main_folder):
    for file in files:
        if file.endswith(".mib") and file != "test.mib":
            full_path = os.path.join(root, file)
            relative_path = os.path.relpath(full_path, main_folder)
            mib_by_name[file] = relative_path

# All .mib files, sorted numerically by filename
mib_files_sorted = sorted(
    mib_by_name.values(),
    key=lambda path: int(
        os.path.splitext(os.path.basename(path))[0]
    )
)

# Files corresponding to dm3_files_sorted_1
mib_files_sorted_1 = [
    mib_by_name[os.path.splitext(dm3_file)[0] + ".mib"]
    for dm3_file in dm3_files_sorted_1
]

# Files corresponding to dm3_files_sorted_2
mib_files_sorted_2 = [
    mib_by_name[os.path.splitext(dm3_file)[0] + ".mib"]
    for dm3_file in dm3_files_sorted_2
]

# This is not AI, useful to write tests to check AI output
assert(len(dm3_files_sorted_1) == len(mib_files_sorted_1))
print(mib_files_sorted)
print()
print(type(mib_files_sorted[0]))
print()
print(mib_files_sorted_1)

# Processing

TL;DR: Notebook 02 with a for loop

In [ ]:
if not os.path.isdir("data"):
    os.mkdir("data")

for file_index in range(len(dm3_files_sorted_1)):
    mib_file = main_folder+mib_files_sorted[file_index]
    dm3_file = main_folder+dm3_files_sorted[file_index]
    s_adf = hs.load(dm3_file)
    x_akse_adf = s_adf.axes_manager[0]
    y_akse_adf = s_adf.axes_manager[1]
    s_4d = hs.load(
        mib_file,
        lazy=True,
        navigation_shape=(x_akse_adf.size+2, y_akse_adf.size)
    )
    s_4d_crop = s_4d.inav[2:]
    
    # TODO: The scale here may need calibatration as in the first notebook
    s_4d_crop.axes_manager[0].scale = x_akse_adf.scale
    s_4d_crop.axes_manager[0].units = x_akse_adf.units
    s_4d_crop.axes_manager[1].scale = y_akse_adf.scale
    s_4d_crop.axes_manager[1].units = y_akse_adf.units
    s_4d_crop.axes_manager[0].name = "Probe x"
    s_4d_crop.axes_manager[1].name = "Probe y"
    s_4d_crop.axes_manager[2].name = "Diffraction x"
    s_4d_crop.axes_manager[3].name = "Diffraction y"
    
    # I think this cropping is good.
    # Found by mousing over a plot before setting scale
    s_4d_crop2 = s_4d_crop.inav[2:-2, 34:500]
    store = zarr.ZipStore(f"data/{file_index}.zspy")
    s_4d_crop2.save(store, chunks=(64,64,64,64))